Maybe try to read in the opera datasets into MintPy for improved time series velocities and ascending/descending decomposition? Then clip the datasets by the aquifers for zonal statistics?

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

from mintpy import view, tsview
import os
from pathlib import Path
import matplotlib.pyplot as plt
import geopandas as gpd

from mintpy.cli.tsview import cmd_line_parse
from mintpy.tsview import timeseriesViewer
from mintpy.view import viewer
from mintpy import geocode
from mintpy import info
import h5py

import rasterio as rio

import rioxarray as rxr
import xarray as xr

In [ ]:
gdf = gpd.read_file(Path(os.getcwd()).parent.parent / "raw_data/TBA_full.shp")


data_storage = Path('/home/clayc/Documents/US_Mex_InSAR') / "OPERA_data"
mintpy_paths = list(data_storage.glob('*/mintpy'))
opera_product_paths = list(data_storage.glob('*/subset-ncs/*.nc'))

In [ ]:
# test = Path('/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy')

test = mintpy_paths[23]


# list the prepared h5 files for use in MintPy
os.listdir(test)

In [ ]:
import rasterio as rio
import numpy as np
from mintpy.utils.utils0 import calc_azimuth_from_east_north_obs, azimuth2heading_angle

los_east = rio.open(test.parent / 'geom_data' / 'los_east.tif').read(1).flatten()
los_north = rio.open(test.parent / 'geom_data' / 'los_north.tif').read(1).flatten()

test_az_angle = calc_azimuth_from_east_north_obs(los_east, los_north)       # azimuth angle

# Calculate up component from unit vector constraint
up = np.sqrt(1 - los_east**2 - los_north**2)

# Incidence angle is angle from vertical
test_inc_angle = np.degrees(np.arccos(up))

In [ ]:
from mintpy import asc_desc2horz_vert

asc_desc2horz_vert.asc_desc2horz_vert(dlos, los_inc_angle, los_az_angle, horz_az_angle=-90, step=20):

In [ ]:
test_ts_disp = h5py.File('/home/clayc/Documents/US_Mex_InSAR/OPERA_data/1092/mintpy/timeseries.h5', 'r')
list(test_ts_disp.keys())
# test_ts_disp['timeseries'][:]

In [ ]:
test_clsc = h5py.File(data_storage / '1092' / 'orbit_data' / 'OPERA_L2_CSLC-S1-STATIC_T005-008737-IW3_20140403_S1A_v1.0.h5', 'r')
test_clsc.keys()

#### Tentatively, I will be skipping the additional ionospheric correction that may be implemented in MintPy. The needed information for this may not be available(?) 

In [ ]:
east_tif = test.parent / 'geom_data' / 'los_east.tif'
north_tif = test.parent / 'geom_data' / 'los_north.tif'
ref_h5 = test / 'timeseries.h5'

create_geom_h5_with_ref(east_tif, north_tif, ref_h5)

In [ ]:
test_geom_h5 = h5py.File(str(ref_h5).replace('timeseries', 'geom'), 'r')
list(test_geom_h5.keys())

In [ ]:
np.unique(test_geom_h5['azimuthAngle'][:])

# Step 1 Decompose Ascending and Descending tracks into horizontal and vertical displacements from current LOS
- asc_des2horz_vert.py

In [ ]:
from mintpy import asc_desc2horz_vert
from mintpy.utils import readfile

In [ ]:
# testing the asc_des2horz_vert on the displacement dataset first
# will then test on the velocity

# TODO: check how to integrate the recommended mask 

# TODO: co-kriging between GNSS and InSAR for filling in masked/incoherent pixels as another project? May havbe to compare NN, linear, non-linear, and ML interpolation methods with co-kriging


ts_atr_list = [readfile.read_attribute(path / 'timeseries.h5') for path in mintpy_paths]
ts_density_atr_list = [readfile.read_attribute(path / 'timeseries_density.h5') for path in mintpy_paths]
vel_atr_list = [readfile.read_attribute(path / 'velocity.h5') for path in mintpy_paths]
mask_atr_list = [readfile.read_attribute(path / 'recommended_mask_90thresh.h5') for path in mintpy_paths]
coh_atr_list = [readfile.read_attribute(path / 'avgSpatialCoh.h5') for path in mintpy_paths]

In [ ]:
test_bbox = asc_desc2horz_vert.get_overlap_lalo(ts_atr_list)

In [ ]:
ts_files = [path / 'timeseries.h5' for path in mintpy_paths]

In [ ]:
asc_desc2horz_vert.run_asc_desc2horz_vert(ts_files)

# Step 2 Mosaick frames together
- image_stitch.py

In [ ]:


        # cmd = 'timeseries.h5 --yx 273 271 --figsize 8 4'
        # inps = cmd_line_parse(cmd.split())
        # obj = timeseriesViewer(inps)
        # obj.open()
        # obj.plot()

# Step 3 Save results to other data formats
- geotiffs, netcdfs, or h5s of average vertical velocity, as well as the timeseries stacks

In [ ]:
cmd = f'view.py {str(test)}/timeseries.h5'
inps = cmd_line_parse(cmd.split()[1:])
obj = viewer()
obj.configure(inps)
obj.plot()

In [ ]:
tsview([str(test / 'velocity.h5')])